## Store vector to Postgres

In [ ]:
%pip install langchain langchain-docling

In [ ]:
%pip install langchain-postgres

In [ ]:
%pip install langchain-core langchain-ollama sqlalchemy

### Prepare Ollama

Since this example use Ollama embedding, we need prepare Ollama service ahead.

```bash
brew install ollama
brew services start ollama
ollama pull llama3
```

### Load env

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os
print(os.getenv('POSTGRES_PORT'))
print(os.getenv('POSTGRES_USER'))
print(os.getenv('TEST'))

### Postgres connection string

In [ ]:
import os

print(os.getenv('POSTGRES_PORT'))
CONNECTION_STRING = (
    f"postgresql+asyncpg://{os.getenv('POSTGRES_USER')}:{os.getenv('POSTGRES_PASSWORD')}@{os.getenv('POSTGRES_HOST')}"
    f":{os.getenv('POSTGRES_PORT')}/{os.getenv('POSTGRES_DB')}"
)

### Create Engine for store, and init table

In [ ]:
from langchain_postgres import PGEngine

pg_engine = PGEngine.from_connection_string(url=CONNECTION_STRING)

await pg_engine.ainit_vectorstore_table(
    table_name=os.getenv('TABLE_NAME'),
    vector_size=os.getenv('VECTOR_SIZE'),
)

### Create Embedding for store

In [12]:
from langchain_ollama import OllamaEmbeddings

embedding = OllamaEmbeddings(model="llama3")

### Create store

In [ ]:
from langchain_postgres import PGVectorStore

store = await PGVectorStore.create(
    engine=pg_engine,
    table_name=os.getenv('TABLE_NAME'),
    # schema_name=SCHEMA_NAME,
    embedding_service=embedding,
)